# Критический анализ коэффициентов CH_zr, CE_zr, CH_EH, CE_EH, CB_EB, CE_EB в режимах TE, TM и гибридном

## 1. Физический смысл коэффициентов

Коэффициенты представляют собой **отношения поперечных полей к продольным** в скин-слое проводника:

| Коэффициент | Определение | Мода | Смысл |
|---|---|---|---|
| `CH_zr` / `CH_zl` | H_y / E_z | TM | отношение магнитного к электрическому |
| `CE_zr` / `CE_zl` | E_y / H_z | TE | отношение электрического к магнитному |
| `CH_EH_l/r` | аналитически вычисленное значение `CH_zr` | TM | из полного решения Maxwell |
| `CE_EH_l/r` | аналитически вычисленное значение `CE_zr` | TE | из полного решения Maxwell |
| `CB_EB_l/r` | B_y / E_z  или  E_y / B_z | TM/TE | то же через B вместо H |
| `CE_EB_l/r` | E_y / B_z  или  E_y / B_z | TE/TM | то же через B вместо H |

Эти коэффициенты используются в упрощённом представлении полей в скин-слое (ячейки 599–600):

```python
# TM:
H_y_r(x) = B_E_zr * CH_zr * exp(-I*K_E_r*(a - x))    # ← H_y пропорционально E_z

# TE:
E_y_r(x) = B_H_zr * CE_zr * exp(-I*K_H_r*(a - x))    # ← E_y пропорционально H_z
```

## 2. Происхождение поперечных полей из уравнений Максвелла

Поперечные компоненты H_y и E_y выводятся из пар уравнений Максвелла методом исключения (ячейки 353–370, функция `get_field`):

Из уравнений 2 и 5 (в нотации тетрадки):

```
уравнение 2:  iωμ'_yy/c · H_y  −  iK_z · E_x  =  −∂E_z/∂x
уравнение 5: −iωε'_xx/c · E_x  −  iK_z · H_y  =  −∂H_z/∂x
```

Решая эту систему 2×2 относительно H_y (исключая E_x):

$$H_y = \frac{1}{\Delta} \left[ \frac{\omega \epsilon'_{xx}}{c} \cdot \frac{\partial E_z}{\partial x} + i K_z \cdot \frac{\partial H_z}{\partial x} \right]$$

где $\Delta = \frac{\omega^2 \epsilon'_{xx} \mu'_{yy}}{c^2} - K_z^2$

**Ключевое наблюдение: H_y зависит от ОБОИХ продольных полей E_z и H_z!**

Аналогично для E_y:

$$E_y = \frac{1}{\Delta'} \left[ \frac{\omega \mu'_{xx}}{c} \cdot \frac{\partial H_z}{\partial x} + i K_z \cdot \frac{\partial E_z}{\partial x} \right]$$

## 3. Режим a) — чистый TE (is_TM=False, is_TE=True)

**E_z = 0** (продольная электрическая компонента отсутствует).

Подставляя E_z = 0 и H_z = B_H · exp(−iK_H·x'):

$$H_y = \frac{iK_z}{\Delta} \cdot (-iK_H) \cdot B_H \cdot e^{-iK_H x'} = \frac{K_z K_H}{\Delta} \cdot B_H \cdot e^{-iK_H x'}$$

$$E_y = \frac{\omega\mu'_{xx}/c}{\Delta'} \cdot (-iK_H) \cdot B_H \cdot e^{-iK_H x'} = \text{const} \cdot B_H \cdot e^{-iK_H x'}$$

**Отношение CE_EH = E_y / H_z:**

$$\frac{E_y}{H_z} = \frac{\text{const} \cdot B_H \cdot e^{-iK_H x'}}{B_H \cdot e^{-iK_H x'}} = \text{const}$$

✅ **Коэффициент CE_zr — истинная константа, не зависящая от x**

✅ Функция `simplify_CEH` работает корректно (экспоненты сокращаются, результат — произведение)

✅ Упрощённое представление `E_y_r(x) = B_H_zr * CE_zr * exp(−iK_H_r*(a−x))` — **точное**

## 4. Режим b) — чистый TM (is_TM=True, is_TE=False)

**H_z = 0** (продольная магнитная компонента отсутствует).

Подставляя H_z = 0 и E_z = B_E · exp(−iK_E·x'):

$$H_y = \frac{\omega\epsilon'_{xx}/c}{\Delta} \cdot (-iK_E) \cdot B_E \cdot e^{-iK_E x'} = \text{const} \cdot B_E \cdot e^{-iK_E x'}$$

**Отношение CH_EH = H_y / E_z:**

$$\frac{H_y}{E_z} = \frac{\text{const} \cdot B_E \cdot e^{-iK_E x'}}{B_E \cdot e^{-iK_E x'}} = \text{const}$$

✅ **Коэффициент CH_zr — истинная константа, не зависящая от x**

✅ Функция `simplify_CEH` работает корректно

✅ Упрощённое представление `H_y_r(x) = B_E_zr * CH_zr * exp(−iK_E_r*(a−x))` — **точное**

## 5. Режим c) — гибридный (is_TM=True, is_TE=True, is_4x4=False)

### 5.1. Структура полных полей

При is_4x4=False (ячейка 352) **ни E_z, ни H_z не обнуляются**:

```
E_z(x) = B_E_zl · exp(−iK_E · (x+a))     # TM часть
H_z(x) = B_H_zl · exp(−iK_H · (x+a))     # TE часть
```

где K_E = √(K²_x_TM) и K_H = √(K²_x_TE) — **разные** волновые числа из разных дисперсионных соотношений (ячейки 437–442):

```
K²_x_TM зависит от ε_yy, ε_zz, μ_xx
K²_x_TE зависит от μ_yy, μ_zz, ε_xx
```

Подставляя в формулу для H_y:

$$\boxed{H_y = \underbrace{\alpha_E \cdot B_E \cdot e^{-iK_E x'}}_{\text{от TM: пропорционально E}_z} + \underbrace{\alpha_H \cdot B_H \cdot e^{-iK_H x'}}_{\text{от TE: пропорционально H}_z}}$$

**H_y содержит ДВА слагаемых с РАЗНЫМИ экспонентами!**

Аналогично:

$$\boxed{E_y = \underbrace{\beta_E \cdot B_E \cdot e^{-iK_E x'}}_{\text{от TM: пропорционально E}_z} + \underbrace{\beta_H \cdot B_H \cdot e^{-iK_H x'}}_{\text{от TE: пропорционально H}_z}}$$

### 5.2. ❌ ПРОБЛЕМА 1: Коэффициент CH_EH_l — НЕ константа

В ячейке 606:
```python
CH_EH_l = Hy_left_conductor / E_left_conductor_sol_z(x)
```

Это даёт:

$$\frac{H_y}{E_z} = \frac{\alpha_E \cdot B_E \cdot e^{-iK_E x'} + \alpha_H \cdot B_H \cdot e^{-iK_H x'}}{B_E \cdot e^{-iK_E x'}}$$

$$= \alpha_E + \alpha_H \cdot \frac{B_H}{B_E} \cdot e^{-i(K_H - K_E) x'}$$

**Это НЕ константа!** Это зависит от x через экспоненту $e^{-i(K_H - K_E)x'}$.

Даже в изотропном случае (K_E = K_H), результат:

$$\frac{H_y}{E_z} = \alpha_E + \alpha_H \cdot \frac{B_H}{B_E}$$

Это формально константа, но представляет собой **СУММУ** двух слагаемых.

### 5.3. ❌ ПРОБЛЕМА 2: simplify_CEH ломается на суммах

Функция `simplify_CEH` (ячейка 605):

```python
def simplify_CEH(C1_EH_l):
    C1_EH_ls = 1
    for operand in C1_EH_l.operands():
        if exp == operand.operator():
            continue
        C1_EH_ls *= operand         # ← перемножает операнды!
    return C1_EH_ls
```

Эта функция спроектирована для выражений вида **ПРОИЗВЕДЕНИЕ** `C × exp(...)`:
- `operands()` возвращает [C, exp(...)]
- exp отбрасывается
- Результат: C ✓

**Но в гибридном режиме выражение имеет вид СУММА** `A + B·exp(...)`:
- `operands()` возвращает [A, B·exp(...)]
- Ни один оператор верхнего уровня ≠ exp
- **Результат: A × B·exp(...) — ПРОИЗВЕДЕНИЕ вместо СУММЫ!** ✗

Это фундаментальная структурная ошибка: функция предполагает мультипликативную декомпозицию, а получает аддитивную.

### 5.4. ❌ ПРОБЛЕМА 3: Упрощённое представление полей теряет перекрёстные члены

Код (ячейки 599–600) представляет поля как:

```
TM: H_y_r(x) = B_E_zr · CH_zr · exp(−iK_E_r·(a−x))     # только E_z-вклад
TE: E_y_r(x) = B_H_zr · CE_zr · exp(−iK_H_r·(a−x))     # только H_z-вклад
```

В реальности:
```
H_y = αE·B_E·exp(−iK_E·x') + αH·B_H·exp(−iK_H·x')      # оба вклада!
E_y = βE·B_E·exp(−iK_E·x') + βH·B_H·exp(−iK_H·x')      # оба вклада!
```

**Пропущенные перекрёстные члены:**
- В H_y: член αH·B_H·exp(−iK_H·x') — вклад H_z в поперечное магнитное поле
- В E_y: член βE·B_E·exp(−iK_E·x') — вклад E_z в поперечное электрическое поле

### 5.5. ❌ ПРОБЛЕМА 4: Силовые плотности теряют перекрёстные члены

Плотности сил (ячейки 615–622) строятся аддитивно:

```python
f_je_conj_r = 0
if is_TM:
    f_je_conj_r += σ_e · E_z_r(x) · conj(H_y_r(x))    # TM-часть
if is_TE:
    f_je_conj_r += σ_e · E_y_r(x) · conj(H_z_r(x))    # TE-часть
```

Но физическая плотность силы в гибридном режиме должна использовать ПОЛНЫЕ поля:

$$f \propto \sigma_e \cdot E_z^{\text{full}} \cdot \overline{H_y^{\text{full}}} + \sigma_e \cdot E_y^{\text{full}} \cdot \overline{H_z^{\text{full}}}$$

Раскрывая с полными полями:

$$\sigma_e \cdot E_z \cdot \overline{H_y} = \sigma_e \cdot (B_E e^{-iK_E x'}) \cdot \overline{(\alpha_E B_E e^{-iK_E x'} + \alpha_H B_H e^{-iK_H x'})}$$

$$= \underbrace{\sigma_e |B_E|^2 \overline{\alpha_E} e^{-2\text{Im}(K_E)x'}}_{\text{чистый TM}} + \underbrace{\sigma_e B_E \overline{\alpha_H B_H} e^{-i(K_E - \overline{K_H})x'}}_{\text{ПЕРЕКРЁСТНЫЙ ЧЛЕН}}$$

**Перекрёстный член полностью отсутствует в текущем алгоритме!**

## 6. Итоговая таблица

| Аспект | Чистый TE | Чистый TM | Гибридный |
|---|---|---|---|
| CH_EH = H_y/E_z | не используется | **const** ✅ | **≠ const**, зависит от x ❌ |
| CE_EH = E_y/H_z | **const** ✅ | не используется | **≠ const**, зависит от x ❌ |
| `simplify_CEH` | корректно (произведение) ✅ | корректно (произведение) ✅ | ломается (сумма→произведение) ❌ |
| Упрощ. поля | точные ✅ | точные ✅ | потеряны cross-coupling ❌ |
| Силовая плотность | корректна ✅ | корректна ✅ | потеряны cross-terms ❌ |
| CB_EB, CE_EB | аналогично CE_EH ✅ | аналогично CH_EH ✅ | те же проблемы ❌ |

## 7. Корневая причина

Весь алгоритм расчёта пондеромоторных сил в скин-слое построен на предположении:

> **«Каждое поперечное поле пропорционально ОДНОМУ продольному полю с ОДНИМ волновым числом»**

Это верно ТОЛЬКО для чистых мод:
- TM: H_y = const · E_z  (H_z = 0)
- TE: E_y = const · H_z  (E_z = 0)

В гибридном режиме поперечные поля содержат **суперпозицию** вкладов от обоих продольных полей с **разными** пространственными зависимостями. Факторизация `поперечное_поле = коэффициент × продольное_поле` принципиально невозможна одним числом.

## 8. Рекомендации по исправлению

### Вариант A: Использовать ПОЛНЫЕ аналитические поля
Вместо упрощённых `E_z_r, H_y_r, H_z_r, E_y_r` использовать полные выражения из ячеек 376–389 (`H_left_conductor_sub_y`, `E_left_conductor_sub_y` и т.д.) непосредственно в формулах силовых плотностей.

### Вариант B: Четырёхкомпонентные коэффициенты
Ввести матрицу коэффициентов 2×2:

$$\begin{pmatrix} H_y \\ E_y \end{pmatrix} = \begin{pmatrix} \alpha_E & \alpha_H \\ \beta_E & \beta_H \end{pmatrix} \begin{pmatrix} B_E \cdot e^{-iK_E x'} \\ B_H \cdot e^{-iK_H x'} \end{pmatrix}$$

И учесть все 4 члена в выражениях для сил.

### Вариант C: is_4x4=True (текущий обходной путь)
При is_4x4=True (ячейка 352) в TM-моде принудительно H_z=0, в TE-моде E_z=0, что восстанавливает корректность коэффициентов. Но это не позволяет рассчитывать ОДНОВРЕМЕННО TE+TM в одном прогоне — только последовательно с суммированием результатов, что физически корректно только при отсутствии перекрёстного взаимодействия мод.

Продолжение рассмотрения инженерного алгоритма, используемого в численном  электродинамическом расчёте волнового двигателя. 
		Основная тестовая и расчётная  среда — файл MenDrive_real.ipynb в режиме use_phase_y, для того чтобы проще было считать дивергенцию B и D.

		Твое предложение: Критический анализ коэффициентов... (твое предложение полностью загружено в файле analysis_hybrid_mode.md).

		### Вариант B: Четырёхкомпонентные коэффициенты
		Ввести матрицу коэффициентов 2×2:

		$$\begin{pmatrix} H_y \\ E_y \end{pmatrix} = \begin{pmatrix} \alpha_E & \alpha_H \\ \beta_E & \beta_H \end{pmatrix} \begin{pmatrix} B_E \cdot e^{-iK_E x'} \\ B_H \cdot e^{-iK_H x'} \end{pmatrix}$$

		И учесть все 4 члена в выражениях для сил.

		Мне идея эта нравится, но я думаю, что поскольку кроме H_y \\ E_y нам нужно искать 
		H_x \\ E_x в зависимости от двух продольных компонент поля, то: 

		Предложи матрицу коэффициентов 4 на 2.
		На основе этой матрицы нужно: 
		переработать исходный код функций расчёта пондеромоторных сил, которые на данный момент используют 
		коэффициенты 
		CH_zr, CE_zr, CH_EH, CE_EH, CB_EB, CE_EB
		а будут использовать матрицу коэффициентов 4 на 2.

Давайте детально изучу все компоненты, которые нужно переработать.Now let me also check how the convective and surface force functions handle the fields:Now I have the complete picture. Let me create the comprehensive document.Ключевые моменты предложения:

**Матрица M(4×2)** связывает 4 поперечные компоненты с 2 продольными:

$$\begin{pmatrix} H_x \\ H_y \\ E_x \\ E_y \end{pmatrix} = \begin{pmatrix} m_{Hx,E} & m_{Hx,H} \\ m_{Hy,E} & m_{Hy,H} \\ m_{Ex,E} & m_{Ex,H} \\ m_{Ey,E} & m_{Ey,H} \end{pmatrix} \begin{pmatrix} B_E \cdot e^{-iK_E x'} \\ B_H \cdot e^{-iK_H x'} \end{pmatrix}$$

**Важное открытие при анализе y-зависимостей:** строки {H_y, E_x} имеют cos(k_y·y), строки {H_x, E_y} имеют sin(k_y·y) — и это одинаково для обоих столбцов в каждой строке. Поэтому элементы M — чистые числа (y факторизуется).

**Переделке подлежат только 10 функций Категории A** (интегралы по скин-слою: ячейки 656–676). Поверхностные силы (ячейки 678–698) уже используют полные поля из `subs_fields()` и корректны как есть.

**Элементы M извлекаются** из существующих аналитических полей (ячейки 377–386) подстановкой B_H=0 / B_E=0 — никаких новых выводов формул не нужно.

**Каждый силовой интеграл** раскладывается в 4 члена: EE (чистый TM), HH (чистый TE), EH и HE (перекрёстные). В чистых модах cross-члены автоматически обнуляются → полная обратная совместимость.

см файл matrix_4x2_hybrid_mode.md

# Матрица коэффициентов M(4×2) для гибридного режима MenDrive

## 1. Классификация силовых функций по типу входных полей

Прежде чем строить матрицу, важно разделить все 8 компонент пондеромоторной силы на две категории.

### Категория A: Интегралы по скин-слою (используют упрощённые поля)
**Эти функции сейчас используют скалярные коэффициенты CH/CE и требуют перехода на M(4×2):**

| Ячейка | Функция | Физический смысл | Использует |
|--------|---------|-----------------|------------|
| 656-657 | `calc_F_jmD_conj_left/right` | Сила D на магнитный ток Фуко | CH_zl/r, CE_zl/r |
| 658-659 | `calc_p_Ht_left/right_jeH_conj` | Сила H на электрический ток Фуко | CH_zl/r, CE_zl/r |
| 660-661 | `calc_p_Bt_left/right_jeB_conj` | Сила B на электрический ток Фуко | CB_EB_l/r, CE_EB_l/r |
| 673-675 | `calc_F_I_conj_l/r_d` | Абрагам на намагниченность I | CH_zl/r, CE_zl/r |
| 674-676 | `calc_F_P_conj_l/r_d` | Абрагам на поляризацию P | CH_zl/r, CE_zl/r |

### Категория B: Граничные (поверхностные) силы (используют полные числовые поля)
**Эти функции УЖЕ корректны в гибридном режиме, т.к. используют полные поля из `subs_fields()`:**

| Ячейка | Функция | Входные поля |
|--------|---------|-------------|
| 678-679 | `calc_p_Ht_left/right_conductor_abs` | Hy_v_d, Hz_v_d (полные, числовые) |
| 680-681 | `calc_p_Bt_left/right_conductor_abs` | Hy_v_d, Hz_v_d |
| 682-689 | `calc_p_I_conv_Ht_..._abs/conj` | Hy_v_d, Hz_v_d, Hx_v_d |
| 686-689 | `calc_t_I_conv_Hn_..._abs/conj` | Hx_v_d |
| 691-694 | `calc_p_Et_..._abs/conj` | Ez_v_d, Ey_v_d |
| 695-698 | `calc_t_Dn_..._abs/conj` | Dx_l/r_d, Dx_v_d, Ex_v_d |

**Вывод:** переделке подлежат ТОЛЬКО 10 функций Категории A (5 пар left/right).

---

## 2. Определение матрицы M(4×2)

### 2.1 Физическая основа

Из уравнений Максвелла (ячейки 353–367, функция `get_field`) каждая поперечная
компонента поля выражается через ∂E_z/∂x, ∂H_z/∂x, и (при use_phase_y) ∂E_z/∂y, ∂H_z/∂y.

В проводнике:
```
E_z(x) = B_E · exp(-iK_E·x') · φ_E(y)     ← TM продольная
H_z(x) = B_H · exp(-iK_H·x') · φ_H(y)     ← TE продольная
```

где x' = (a - x) для правого проводника, x' = (a + x) для левого.

Все 4 поперечные компоненты — линейные комбинации двух источников:

$$\begin{pmatrix} H_x \\ H_y \\ E_x \\ E_y \end{pmatrix} = \begin{pmatrix} m_{Hx,E} & m_{Hx,H} \\ m_{Hy,E} & m_{Hy,H} \\ m_{Ex,E} & m_{Ex,H} \\ m_{Ey,E} & m_{Ey,H} \end{pmatrix} \begin{pmatrix} B_E \cdot e^{-iK_E x'} \cdot \psi_1(y) \\ B_H \cdot e^{-iK_H x'} \cdot \psi_2(y) \end{pmatrix}$$

**Важно:** y-зависимости ψ_1 и ψ_2 различаются по строкам:

| Строка матрицы | Тип y-зависимости столбца E | Тип y-зависимости столбца H |
|---|---|---|
| H_x: | sin(k_y·y) от ∂E_z/∂y | sin(k_y·y) от ∂H_z/∂x·(phase_y_TE) |
| H_y: | cos(k_y·y) от ∂E_z/∂x | cos(k_y·y) от ∂H_z/∂y |
| E_x: | cos(k_y·y) от ∂E_z/∂x | cos(k_y·y) от ∂H_z/∂y |
| E_y: | sin(k_y·y) от ∂E_z/∂y | sin(k_y·y) от ∂H_z/∂x |

Это означает, что в каждой строке оба элемента имеют **одинаковую** y-зависимость, что позволяет вынести её за скобки. Факторизация по y работает.

### 2.2 Формулы элементов

Обозначим (для правого проводника, ∂/∂x → производная действует на exp(-iK·x')):

$$Q_1 = \frac{\omega^2 \mu'_{yy}\epsilon'_{xx}}{c^2} - k_z^2 \qquad Q_2 = \frac{\omega^2 \mu'_{xx}\epsilon'_{yy}}{c^2} - k_z^2$$

**Столбец E (от E_z):**

Из ячеек 353–362 (вывод через get_field: решение пар уравнений Максвелла):

$$m_{Hy,E} = \frac{-i K_E \cdot \omega\epsilon'_{xx}/c}{Q_1} \qquad (\text{из пары eq 2+5, ∂E_z/∂x вклад})$$

$$m_{Ex,E} = \frac{i K_E \cdot k_z}{Q_1} \qquad (\text{из пары eq 2+5, ∂E_z/∂x вклад})$$

$$m_{Hx,E} = \frac{k_{y,E} \cdot k_z}{Q_2} \qquad (\text{из пары eq 1+4, ∂E_z/∂y вклад})$$

$$m_{Ey,E} = \frac{-i k_{y,E} \cdot \omega\mu'_{xx}/c}{Q_2} \qquad (\text{из пары eq 1+4, ∂E_z/∂y вклад})$$

(здесь k_{y,E} — коэффициент при y-производной φ_E(y): для cos(k_y·y) → dφ_E/dy = -k_y·sin → k_{y,E} = -k_y)

**Столбец H (от H_z):**

$$m_{Hy,H} = \frac{-i k_{y,H} \cdot k_z}{Q_1} \qquad (\text{из пары eq 2+5, ∂H_z/∂y вклад})$$

$$m_{Ex,H} = \frac{i k_{y,H} \cdot \omega\mu'_{yy}/c}{Q_1} \qquad (\text{из пары eq 2+5, ∂H_z/∂y вклад})$$

$$m_{Hx,H} = \frac{i K_H \cdot \omega\epsilon'_{yy}/c}{Q_2} \qquad (\text{из пары eq 1+4, ∂H_z/∂x вклад})$$

$$m_{Ey,H} = \frac{-i K_H \cdot k_z}{Q_2} \qquad (\text{из пары eq 1+4, ∂H_z/∂x вклад})$$

(здесь k_{y,H} — коэффициент при y-производной φ_H(y): для sin(k_y·y) → dφ_H/dy = k_y·cos → k_{y,H} = +k_y)

**Примечания:**
- Знаки при K_E, K_H зависят от направления ∂/∂x → ±iK (правый vs левый проводник)
- Знаки при k_y зависят от выбора is_cos_y_TM, is_cos_y_TE
- В формулах выше k_z = k_z + i·s_z (комплексное)

### 2.3 Матрица для B и D

$$\mathbf{M}^{(B)} = \begin{pmatrix} \mu'_{xx} \cdot m_{Hx,E} & \mu'_{xx} \cdot m_{Hx,H} \\ \mu'_{yy} \cdot m_{Hy,E} & \mu'_{yy} \cdot m_{Hy,H} \end{pmatrix}$$

$$\mathbf{M}^{(D)} = \begin{pmatrix} \epsilon'_{xx} \cdot m_{Ex,E} & \epsilon'_{xx} \cdot m_{Ex,H} \\ \epsilon'_{yy} \cdot m_{Ey,E} & \epsilon'_{yy} \cdot m_{Ey,H} \end{pmatrix}$$

### 2.4 Связь со старыми скалярными коэффициентами

При B_H = 0 (чистый TM):
```
CH_EH = m_{Hy,E}      (= Hy/Ez при Hz=0)
CB_EB = μ'_yy · m_{Hy,E}
```

При B_E = 0 (чистый TE):
```
CE_EH = m_{Ey,H}      (= Ey/Hz при Ez=0)
CE_EB = m_{Ey,H} / μ'_zz   (через Bz = μ'_zz·Hz)
```

---

## 3. Извлечение элементов матрицы из существующего кода

Аналитические выражения полей **уже вычислены** в ячейках 377–386:

```python
# Ячейка 377: H_left_conductor_sub_x(x) — содержит ОБА слагаемых (от E_z и H_z)
# Ячейка 378: H_right_conductor_sub_x(x)
# Ячейка 380: H_left_conductor_sub_y(x)   → Hy_left_conductor (ячейка 489)
# Ячейка 381: H_right_conductor_sub_y(x)
# Ячейка 385: E_left_conductor_sub_x(x)
# Ячейка 386: E_right_conductor_sub_x(x)
# Ячейка 381: E_left_conductor_sub_y(x)
# Ячейка 382: E_right_conductor_sub_y(x)
```

**Метод извлечения элементов M:**

Каждое выражение вида `H_left_conductor_sub_y(x)` в гибридном режиме имеет структуру:

```
H_left_conductor_sub_y(x) = α · B_E_zl · exp(-iK_E_l·(a+x)) · φ_E(y)
                            + β · B_H_zl · exp(-iK_H_l·(a+x)) · φ_H(y)
```

Чтобы извлечь элементы матрицы, делим на соответствующий продольный множитель:

```python
# НОВАЯ функция вместо simplify_CEH
def extract_M_element_E(field_expr, side='left'):
    """Извлекает элемент столбца E матрицы M (коэффициент при B_E·exp(-iK_E·x'))"""
    if side == 'left':
        return (field_expr / (E_left_conductor_sol_z(x) * phase_y_TM)).simplify()
    else:
        return (field_expr / (E_right_conductor_sol_z(x) * phase_y_TM)).simplify()

def extract_M_element_H(field_expr, side='left'):
    """Извлекает элемент столбца H матрицы M (коэффициент при B_H·exp(-iK_H·x'))"""
    if side == 'left':
        return (field_expr / (H_left_conductor_sol_z(x) * phase_y_TE)).simplify()
    else:
        return (field_expr / (H_right_conductor_sol_z(x) * phase_y_TE)).simplify()
```

**НО!** Это деление НЕ даст чистый элемент в гибридном режиме (получится сумма + перекрёстный член). Нужен другой подход:

### 3.1 Правильный метод: подстановка B_E=1,B_H=0 и B_E=0,B_H=1

```python
def calc_M_matrix(side='left'):
    """Вычисляет матрицу M(4×2) для указанного проводника."""

    if side == 'left':
        Hx_full = H_left_conductor_sub_x(x)
        Hy_full = H_left_conductor_sub_y(x)   # = Hy_left_conductor
        Ex_full = E_left_conductor_sub_x(x)
        Ey_full = E_left_conductor_sub_y(x)
        B_E_sym = B_E_zl
        B_H_sym = B_H_zl
        exp_E   = exp(-I*K_E_l*(a + x))
        exp_H   = exp(-I*K_H_l*(a + x))
    else:
        Hx_full = H_right_conductor_sub_x(x)
        Hy_full = H_right_conductor_sub_y(x)
        Ex_full = E_right_conductor_sub_x(x)
        Ey_full = E_right_conductor_sub_y(x)
        B_E_sym = B_E_zr
        B_H_sym = B_H_zr
        exp_E   = exp(-I*K_E_r*(a - x))
        exp_H   = exp(-I*K_H_r*(a - x))

    fields = [Hx_full, Hy_full, Ex_full, Ey_full]
    field_names = ['Hx', 'Hy', 'Ex', 'Ey']

    M = {}

    for name, field in zip(field_names, fields):
        # Столбец E: подставляем B_H=0, делим на B_E·exp_E
        field_E = field.subs(B_H_sym == 0)
        m_E = (field_E / (B_E_sym * exp_E)).simplify()  # должна остаться const · φ(y)
        M[name, 'E'] = m_E

        # Столбец H: подставляем B_E=0, делим на B_H·exp_H
        field_H = field.subs(B_E_sym == 0)
        m_H = (field_H / (B_H_sym * exp_H)).simplify()
        M[name, 'H'] = m_H

    return M
```

Этот метод гарантирует корректное извлечение, т.к. при B_H=0 остаётся только TM-вклад с одной экспонентой, которая точно сокращается.

### 3.2 Символьное вычисление элементов (замена ячеек 610-612)

```python
# =====================================================
# ЗАМЕНА ЯЧЕЕК 610-612: Вычисление матрицы M(4×2)
# =====================================================

# --- Левый проводник ---
# Столбец E (TM-вклад): подставляем B_H_zl = 0
M_Hx_E_ls = (H_left_conductor_sub_x(x).subs(B_H_zl == 0) \
            / (E_left_conductor_sol_z(x) * phase_y_ratio_Hx)) \
            .subs(sign_K_1_subs)  # phase_y_ratio_Hx = sin(ky·y)/cos(ky·y) и т.д.

M_Hy_E_ls = (H_left_conductor_sub_y(x).subs(B_H_zl == 0) \
            / (E_left_conductor_sol_z(x) * 1)) \
            .subs(sign_K_1_subs)

M_Ex_E_ls = (E_left_conductor_sub_x(x).subs(B_H_zl == 0) \
            / (E_left_conductor_sol_z(x) * 1)) \
            .subs(sign_K_1_subs)

M_Ey_E_ls = (E_left_conductor_sub_y(x).subs(B_H_zl == 0) \
            / (E_left_conductor_sol_z(x) * phase_y_ratio_Ey)) \
            .subs(sign_K_1_subs)

# Столбец H (TE-вклад): подставляем B_E_zl = 0
M_Hx_H_ls = (H_left_conductor_sub_x(x).subs(B_E_zl == 0) \
            / (H_left_conductor_sol_z(x) * phase_y_ratio_Hx)) \
            .subs(sign_K_1_subs)

M_Hy_H_ls = (H_left_conductor_sub_y(x).subs(B_E_zl == 0) \
            / (H_left_conductor_sol_z(x) * phase_y_ratio_Hy)) \
            .subs(sign_K_1_subs)

# ... аналогично для M_Ex_H_ls, M_Ey_H_ls

# --- Правый проводник --- (аналогично с заменой l→r, (a+x)→(a-x))

# Формируем символьные матрицы
# M_left_s  = matrix из M_*_*_ls
# M_right_s = matrix из M_*_*_rs
```

**Примечание о phase_y_ratio:**

При use_phase_y, y-зависимость различается для разных строк:
- H_y и E_x содержат ту же φ(y), что E_z: `cos(k_y·y)` → ratio = 1
- H_x и E_y содержат `sin(k_y·y)` → ratio = `sin(k_y·y)/cos(k_y·y)` при делении на E_z

Но при извлечении через подстановку B_H=0, мы делим на `B_E·exp(-iK_E·x')·φ_E(y)`,
и результат содержит `φ_component(y)/φ_E(y)`:
- Для H_y: `cos(ky·y)/cos(ky·y) = 1` ✓
- Для H_x: `(-sin(ky·y))/cos(ky·y)` — не сокращается!

**Решение:** элемент матрицы определяется БЕЗ y-множителя. Мы вводим:

```python
# Элементы матрицы — чистые числа (без y)
# y-зависимость будет восстановлена при сборке полных полей

m_Hy_E = (Hy_full.subs(B_H==0) / (B_E * exp_E * phase_y_TM))  # ←/cos → чистое число
m_Hx_E = (Hx_full.subs(B_H==0) / (B_E * exp_E * phase_y_Hx))  # ←/sin → чистое число
```

где `phase_y_Hx` = `−sin(k_y·y)` (для данного is_cos_y_TM=True).

---

## 4. Новое представление полей в скин-слое (замена ячеек 604-605)

### 4.1 Полная система (6 компонент)

Для правого проводника:

```python
# =====================================================
# ЗАМЕНА ЯЧЕЕК 604-605: Полные поля через матрицу M
# =====================================================

# Продольные (без изменений):
E_z_r(x) = B_E_zr * exp(-I*K_E_r*(a - x)) * phase_y_TM    # как было
H_z_r(x) = B_H_zr * exp(-I*K_H_r*(a - x)) * phase_y_TE    # как было

# Поперечные (НОВОЕ: два слагаемых каждое):
H_y_r(x) = M_Hy_E_r * B_E_zr * exp(-I*K_E_r*(a-x)) * phase_y_TM \
          + M_Hy_H_r * B_H_zr * exp(-I*K_H_r*(a-x)) * phase_y_Hy_from_H
    # phase_y_Hy_from_H = cos(k_y·y)·(k_y/k_y) — уточнить по ∂H_z/∂y

E_y_r(x) = M_Ey_E_r * B_E_zr * exp(-I*K_E_r*(a-x)) * phase_y_Ey_from_E \
          + M_Ey_H_r * B_H_zr * exp(-I*K_H_r*(a-x)) * phase_y_TE
    # phase_y_Ey_from_E = -sin(k_y·y)·k_y — уточнить по ∂E_z/∂y

H_x_r(x) = M_Hx_E_r * B_E_zr * exp(-I*K_E_r*(a-x)) * phase_y_Hx_from_E \
          + M_Hx_H_r * B_H_zr * exp(-I*K_H_r*(a-x)) * phase_y_TE

E_x_r(x) = M_Ex_E_r * B_E_zr * exp(-I*K_E_r*(a-x)) * phase_y_TM \
          + M_Ex_H_r * B_H_zr * exp(-I*K_H_r*(a-x)) * phase_y_Ex_from_H

# Аналогичные B-компоненты:
B_y_r(x) = mu_r_yy_c * H_y_r(x)
B_x_r(x) = mu_r_xx_c * H_x_r(x)
```

### 4.2 Упрощение: одинаковые y-зависимости в парах

При is_cos_y_TM=True, is_cos_y_TE=False:
- phase_y_TM = cos(k_y_E·y)
- phase_y_TE = sin(k_y_H·y)

Дифференцирование:
- ∂(cos)/∂y = −k_y·sin → sin-тип
- ∂(sin)/∂y = +k_y·cos → cos-тип

Поэтому:

| Поле | Вклад от E_z (∂/∂x → cos) | Вклад от E_z (∂/∂y → sin) | Вклад от H_z (∂/∂x → sin) | Вклад от H_z (∂/∂y → cos) |
|------|---|---|---|---|
| H_y | cos (∂E_z/∂x) ✓ | — | — | cos (∂H_z/∂y) ✓ |
| E_x | cos (∂E_z/∂x) ✓ | — | — | cos (∂H_z/∂y) ✓ |
| H_x | — | sin (∂E_z/∂y) ✓ | sin (∂H_z/∂x) ✓ | — |
| E_y | — | sin (∂E_z/∂y) ✓ | sin (∂H_z/∂x) ✓ | — |

**Строки {H_y, E_x} — cos-тип, строки {H_x, E_y} — sin-тип.**

Это значит, что **в каждой строке y-зависимость одинакова для обоих столбцов**, и матрица M(4×2) действительно состоит из чистых (y-независимых) числовых коэффициентов!

### 4.3 Финальное компактное представление

```python
# cos-группа:
phi_cos = phase_y_TM   # = cos(k_y·y)
# sin-группа:
phi_sin = phase_y_TE   # = sin(k_y·y)

# Продольные:
E_z_r(x,y) = B_E_zr * exp_E_r * phi_cos
H_z_r(x,y) = B_H_zr * exp_H_r * phi_sin

# Поперечные:
H_y_r(x,y) = (M_Hy_E_r * B_E_zr * exp_E_r + M_Hy_H_r * B_H_zr * exp_H_r) * phi_cos
E_x_r(x,y) = (M_Ex_E_r * B_E_zr * exp_E_r + M_Ex_H_r * B_H_zr * exp_H_r) * phi_cos

H_x_r(x,y) = (M_Hx_E_r * B_E_zr * exp_E_r + M_Hx_H_r * B_H_zr * exp_H_r) * phi_sin
E_y_r(x,y) = (M_Ey_E_r * B_E_zr * exp_E_r + M_Ey_H_r * B_H_zr * exp_H_r) * phi_sin
```

где `exp_E_r = exp(-I*K_E_r*(a-x))`, `exp_H_r = exp(-I*K_H_r*(a-x))`.

---

## 5. Переработка формул силовых плотностей (замена ячеек 620-628)

### 5.1 Общий принцип: произведение двух полных полей

Каждая силовая плотность имеет вид `F1(x) · conj(F2(x))`, где F1 и F2 — различные компоненты поля. При раскрытии через M получаем **4 члена**:

```
F1 · conj(F2) = (m1_E·B_E·exp_E + m1_H·B_H·exp_H) · conj(m2_E·B_E·exp_E + m2_H·B_H·exp_H)

= m1_E·conj(m2_E)·|B_E|²·exp(-2·Im(K_E)·x')           ← EE (чистый TM)
+ m1_H·conj(m2_H)·|B_H|²·exp(-2·Im(K_H)·x')           ← HH (чистый TE)
+ m1_E·conj(m2_H)·B_E·conj(B_H)·exp(-i(K_E-conj(K_H))·x')  ← EH (cross)
+ m1_H·conj(m2_E)·B_H·conj(B_E)·exp(-i(K_H-conj(K_E))·x')  ← HE (cross)
```

### 5.2 Типы интегралов по скин-слою

Все интегралы сводятся к трём типам:

```python
# Тип 1: чистый TM
I_EE = integral(exp(-2*Im(K_E)*x'), 0, ∞) = 1 / (2*Im(K_E))

# Тип 2: чистый TE
I_HH = integral(exp(-2*Im(K_H)*x'), 0, ∞) = 1 / (2*Im(K_H))

# Тип 3: перекрёстный (cross)
I_EH = integral(exp(-i*(K_E - conj(K_H))*x'), 0, ∞) = 1 / (i*(K_E - conj(K_H)))
     = 1 / (i*(ReK_E - ReK_H) - (ImK_E + ImK_H))

# Тип 4: сопряжённый перекрёстный
I_HE = conj(I_EH)
```

Перекрёстные интегралы **сходятся**, т.к. Im(K_E) + Im(K_H) > 0.

### 5.3 Пример: сила f_{je} (ток Фуко × H) — замена ячейки 627

**Было (ячейка 627):**
```python
f_jeH_conj_r = 0
if is_TM:
    f_jeH_conj_r += (-sigma_e_r_zz/2/c * E_z_r(x) * exp(-I*omega_t)
                     * conjugate(H_y_r(x) * exp(-I*omega_t)))
if is_TE:
    f_jeH_conj_r += ( sigma_e_r_yy/2/c * E_y_r(x) * exp(-I*omega_t)
                     * conjugate(H_z_r(x) * exp(-I*omega_t)))
```

**Стало (новый код):**
```python
# Полные поля (6 компонент) через элементы M:
exp_E_r = exp(-I*K_E_r*(a - x))
exp_H_r = exp(-I*K_H_r*(a - x))

# E_z: только TM-экспонента
E_z_full_r = B_E_zr * exp_E_r * phi_cos
# H_z: только TE-экспонента
H_z_full_r = B_H_zr * exp_H_r * phi_sin
# H_y: оба вклада, cos-тип
H_y_full_r = (M_Hy_E_r * B_E_zr * exp_E_r + M_Hy_H_r * B_H_zr * exp_H_r) * phi_cos
# E_y: оба вклада, sin-тип
E_y_full_r = (M_Ey_E_r * B_E_zr * exp_E_r + M_Ey_H_r * B_H_zr * exp_H_r) * phi_sin

# Плотность силы (полная):
f_jeH_conj_r = (
    -sigma_e_r_zz/2/c * E_z_full_r * conjugate(H_y_full_r)
    + sigma_e_r_yy/2/c * E_y_full_r * conjugate(H_z_full_r)
) .subs(epsilon_mu_subs).subs(k_re_im_subs)
```

### 5.4 Пример: Абрагам-подобная f_P — замена ячейки 620

**Было:**
```python
f_P_conj_r = 0
if is_TM:
    f_P_conj_r += (1/2 * (epsilon_r_zz_c-1)/(4*pi) * E_z_r(x)
                   * I*omega/c * conjugate(mu_r_perp_c * H_y_r(x))) ...
if is_TE:
    f_P_conj_r += -(1/2 * (epsilon_r_yy_c-1)/(4*pi) * E_y_r(x)
                    * I*omega/c * conjugate(mu_r_perp_c * H_z_r(x))) ...
```

**Стало:**
```python
f_P_conj_r = (
    1/2 * (epsilon_r_zz_c-1)/(4*pi) * E_z_full_r
        * I*omega/c * conjugate(mu_r_perp_c * H_y_full_r)
    - 1/2 * (epsilon_r_yy_c-1)/(4*pi) * E_y_full_r
        * I*omega/c * conjugate(mu_r_perp_c * H_z_full_r)
).subs(epsilon_mu_subs).subs(k_re_im_subs)
```

### 5.5 Единообразный шаблон для всех 5 пар

Все силовые плотности Категории A имеют вид:

$$f = \alpha_1 \cdot \text{Field}_1 \cdot \overline{\text{Field}_2} + \alpha_2 \cdot \text{Field}_3 \cdot \overline{\text{Field}_4}$$

| Сила | α₁ | Field₁ | Field₂ | α₂ | Field₃ | Field₄ |
|---|---|---|---|---|---|---|
| f_jeH (ячейка 627) | −σ_e,zz/(2c) | E_z | H_y | +σ_e,yy/(2c) | E_y | H_z |
| f_jeB (ячейка 628) | −σ_e,zz/(2c) | E_z | B_y | +σ_e,yy/(2c) | E_y | B_z |
| f_jmD (ячейка 625) | +σ_m,yy/(2c) | H_y | D_z(=ε_zz E_z) | −σ_m,zz/(2c) | H_z | D_y(=ε_yy E_y) |
| f_I (ячейка 622) | (μ'-1)/(8π) | H_y | μ'ε'E_z | −(μ'-1)/(8π) | H_z | μ'ε'E_y |
| f_P (ячейка 620) | (ε'-1)/(8π) | E_z | μ'H_y | −(ε'-1)/(8π) | E_y | μ'H_z |

В каждом случае: Field₁ · conj(Field₂) раскрывается в 4 члена через M, интегрируется по x' аналитически.

---

## 6. Переработка интегрирующих функций (замена ячеек 656-676)

### 6.1 Новая вспомогательная функция: calc_M_matrix_d

```python
def calc_M_matrix_d(K_conductor_subs_d, digit_values, k_z_sol, coeffs_subs, side='left'):
    """
    Вычисляет числовые значения всех 8 элементов матрицы M(4×2).
    Возвращает словарь {('Hx','E'): value, ('Hx','H'): value, ...}
    """
    M_d = {}

    # Символьные элементы (вычислены ранее: M_Hx_E_ls, M_Hx_H_ls, ...)
    M_symbols = {
        ('Hx','E'): M_Hx_E_ls if side=='left' else M_Hx_E_rs,
        ('Hx','H'): M_Hx_H_ls if side=='left' else M_Hx_H_rs,
        ('Hy','E'): M_Hy_E_ls if side=='left' else M_Hy_E_rs,
        ('Hy','H'): M_Hy_H_ls if side=='left' else M_Hy_H_rs,
        ('Ex','E'): M_Ex_E_ls if side=='left' else M_Ex_E_rs,
        ('Ex','H'): M_Ex_H_ls if side=='left' else M_Ex_H_rs,
        ('Ey','E'): M_Ey_E_ls if side=='left' else M_Ey_E_rs,
        ('Ey','H'): M_Ey_H_ls if side=='left' else M_Ey_H_rs,
    }

    for key, M_sym in M_symbols.items():
        M_d[key] = M_sym \
            .subs(epsilon_mu_subs).subs(k_re_im_subs) \
            .subs(K_conductor_subs_d) \
            .subs(coefs_re_im_subs) \
            .subs(coeffs_subs).subs(digit_values).subs(k_z_sol).n()

    return M_d
```

### 6.2 Новая универсальная функция интегрирования

```python
def integrate_skin_force(f_density_symbolic, M_d, K_conductor_subs_d,
                         digit_values, k_z_sol, coeffs_subs, side='left'):
    """
    Универсальная функция для интегрирования силовой плотности по скин-слою.

    f_density_symbolic — символьное выражение плотности, записанное через
    M_Hx_E, M_Hy_E, ..., M_Ey_H (символьные переменные матрицы)
    и B_E_zl/r, B_H_zl/r, K_E_l/r, K_H_l/r.
    """

    # Подставляем числовые значения M
    f_subs = f_density_symbolic
    for key, val in M_d.items():
        sym_name = f'M_{key[0]}_{key[1]}'  # e.g. M_Hy_E
        f_subs = f_subs.subs(var(sym_name) == val)

    # Интегрируем и подставляем
    F_integrated = integrate(f_subs, x, ...)  # пределы зависят от side

    # Усредняем по y
    F_integrated = avg_over_y(F_integrated ...)

    # Подставляем числовые значения
    result = real128(F_integrated
        .subs(epsilon_mu_subs)
        .subs(K_conductor_subs_d)
        .subs(digit_values)
        .subs(k_z_sol)
        .subs([B_z_re_im_subs])
        .subs(coeffs_subs).real())

    return result
```

### 6.3 Пример: новая calc_p_Ht_left_jeH_conj (замена ячейки 658)

```python
def calc_p_Ht_left_jeH_conj(K_left_conductor_subs_d, digit_values, k_z_sol,
                              coeffs_subs, logging=False):
    # 1. Вычисляем матрицу M для левого проводника
    M_d = calc_M_matrix_d(K_left_conductor_subs_d, digit_values,
                          k_z_sol, coeffs_subs, side='left')

    # 2. Берём заранее вычисленный символьный интеграл F_je_conj_l
    #    (который теперь содержит M-элементы вместо CH/CE)
    F_subs = F_je_conj_l  # символьное выражение с M_Hy_E, M_Hy_H, M_Ey_E, M_Ey_H

    # 3. Подставляем числовые значения элементов M
    F_subs = F_subs \
        .subs(M_Hy_E == M_d[('Hy','E')]) \
        .subs(M_Hy_H == M_d[('Hy','H')]) \
        .subs(M_Ey_E == M_d[('Ey','E')]) \
        .subs(M_Ey_H == M_d[('Ey','H')])

    # 4. Разделяем на Re/Im части K для интегрирования
    F_subs = avg_over_y(F_subs
        .subs([K_E_l == ReK_E_l + I*ImK_E_l,
               K_H_l == ReK_H_l + I*ImK_H_l])) \
        .subs([ImK_H_l == K_H_left_conductor.imag(),
               ImK_E_l == K_E_left_conductor.imag()])

    # 5. Числовой результат
    result = real128(F_subs
        .subs(epsilon_mu_subs)
        .subs(K_left_conductor_subs_d)
        .subs(digit_values)
        .subs(k_z_sol)
        .subs([B_zl_re_im_subs])
        .subs(coeffs_subs).real())

    return result
```

---

## 7. Обратная совместимость с чистыми модами

При is_TM=True, is_TE=False:
- B_H_zl = B_H_zr = 0 (из SVD решения)
- M_*_H элементы не используются (все TE-вклады обнуляются)
- Остаются только EE-члены → результат идентичен старому коду с CH_EH

При is_TM=False, is_TE=True:
- B_E_zl = B_E_zr = 0
- M_*_E элементы не используются
- Остаются только HH-члены → результат идентичен старому коду с CE_EH

**Гибридный режим** добавляет:
- EH и HE перекрёстные члены
- Все 8 элементов матрицы задействованы

---

## 8. Полная таблица изменений по ячейкам

| Ячейки | Что сейчас | Что будет | Статус |
|--------|-----------|-----------|--------|
| 604-605 | Упрощённые поля E_z_r, H_y_r, H_z_r, E_y_r | Полные поля через M(4×2) с 6 компонентами | **ПЕРЕПИСАТЬ** |
| 610 | `simplify_CEH()` | Удалить (не нужна) | **УДАЛИТЬ** |
| 611 | CH_EH_l/r, CE_EH_l/r через simplify_CEH | 8 элементов M через подстановку B_E=0/B_H=0 | **ПЕРЕПИСАТЬ** |
| 612 | CB_EB_l/r, CE_EB_l/r через simplify_CEH | Вычислять через M·diag(μ) | **ПЕРЕПИСАТЬ** |
| 613, 615 | `calc_C_EH_ld/rd()` | `calc_M_matrix_d()` — числовые значения M(4×2) | **ПЕРЕПИСАТЬ** |
| 620-628 | Силовые плотности с раздельным if is_TM/is_TE | Полные плотности через full-поля (всегда оба вклада) | **ПЕРЕПИСАТЬ** |
| 626, 629 | Интегралы с одним типом экспоненты | Интегралы с 4 типами экспонент (EE, HH, EH, HE) | **ПЕРЕПИСАТЬ** |
| 656-676 | calc_F_*() с подстановкой CH_zl==CH_EH_ld | calc_F_*() с подстановкой 8 элементов M_d | **ПЕРЕПИСАТЬ** |
| 678-698 | Поверхностные силы через полные поля | **Без изменений** (уже корректны) | ✅ |

---

## 9. Резюме

Матрица M(4×2) — это естественное обобщение скалярных коэффициентов CH/CE:

$$\underbrace{\begin{pmatrix} CH\_EH \end{pmatrix}}_{\text{старый 1×1}} \longrightarrow \underbrace{\begin{pmatrix} m_{Hx,E} & m_{Hx,H} \\ m_{Hy,E} & m_{Hy,H} \\ m_{Ex,E} & m_{Ex,H} \\ m_{Ey,E} & m_{Ey,H} \end{pmatrix}}_{\text{новый 4×2}}$$

- В чистых модах: только один столбец ненулевой → обратно к скаляру
- В гибриде: оба столбца активны → перекрёстные члены учтены
- Элементы извлекаются из **уже вычисленных** аналитических полей подстановкой B_H=0 / B_E=0
- Структура calc_* функций сохраняется, меняется только подстановка: вместо 2 скаляров подставляем 8 элементов матрицы